# Chapter 33 — Reinforcement Learning I: MDPs, Bellman Equation & Value Iteration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part5_production_rl/ch33_reinforcement_learning_i/ch33_notebook.ipynb)

Companion notebook for **Chapter 33** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Reinforcement learning learns from delayed reward by trial and error, with no labeled examples anywhere.
- It suits problems that are easy to evaluate but hard to specify, such as playing chess well.
- An MDP is defined by states, actions, a transition model and a reward function, and the Markov property keeps it tractable.
- The discount factor gamma sets how far ahead the agent cares: near 0 is short-sighted, near 1 is patient.
- The Bellman equation links a state's utility to its neighbors', and value iteration solves it by repeated updates.
- Q-learning reaches the same optimal policy without ever knowing the transition probabilities, learning purely from experience.

## Example 1: Value Iteration on the 4x3 Grid World

This solves the classic 4x3 grid world: wall at (2,2), terminal states +1 and -1, step reward -0.04, and actions succeeding with probability 0.8 while slipping sideways with probability 0.1 each. Applying the Bellman update repeatedly converges in 22 iterations to the published utilities - U(1,1)=0.705, U(1,3)=0.812, U(3,3)=0.918 - and the extracted policy routes the agent safely around the wall.

In [1]:
# Chapter 33 - Example 1: Value Iteration on the 4x3 Grid World
import numpy as np
np.set_printoptions(suppress=True)

print("=== Value Iteration on the 4x3 Grid World ===\n")
print("Grid (column, row), (1,1) is bottom-left:\n")
print("     3 | . . . +1")
print("     2 | . # . -1        # = wall/forbidden square (2,2)")
print("     1 | S . . .         S = START")
print("       +---------")
print("       1 2 3 4\n")
print("Stochastic environment: the intended action succeeds with p=0.8,")
print("and moves at right angles with p=0.1 each. Bumping into a wall")
print("leaves the agent in the same square.")
print("Reward: -0.04 for every non-terminal transition. Discount gamma=1.0\n")

rows, cols = 3, 4
WALL      = (2, 2)
TERMINALS = {(4, 3): +1.0, (4, 2): -1.0}
R_STEP    = -0.04
GAMMA     = 1.0

states = [(c, r) for c in range(1, cols+1) for r in range(1, rows+1)
          if (c, r) != WALL]
ACTIONS = {"Up": (0, 1), "Down": (0, -1), "Left": (-1, 0), "Right": (1, 0)}
RIGHT_ANGLES = {"Up": ["Left", "Right"], "Down": ["Left", "Right"],
                "Left": ["Up", "Down"],  "Right": ["Up", "Down"]}

def move(s, a):
    dc, dr = ACTIONS[a]
    ns = (s[0] + dc, s[1] + dr)
    if ns == WALL or not (1 <= ns[0] <= cols and 1 <= ns[1] <= rows):
        return s                      # bumped into a wall -> stay
    return ns

def transitions(s, a):
    """Returns list of (probability, next_state)."""
    out = [(0.8, move(s, a))]
    for ra in RIGHT_ANGLES[a]:
        out.append((0.1, move(s, ra)))
    return out

# ---- Value iteration ----
U = {s: 0.0 for s in states}
for s, v in TERMINALS.items():
    U[s] = v

print("Bellman update:  U(s) <- max_a  sum_s' P(s'|s,a) [ R(s,a,s') + gamma*U(s') ]\n")
print(f"{'iter':>5}{'U(1,1)':>10}{'U(1,3)':>10}{'U(3,3)':>10}{'U(4,1)':>10}{'max change':>13}")
print("-" * 58)

for it in range(1, 60):
    U_new = dict(U)
    delta = 0.0
    for s in states:
        if s in TERMINALS:
            continue
        best = -1e9
        for a in ACTIONS:
            q = sum(p * (R_STEP + GAMMA * U[ns]) for p, ns in transitions(s, a))
            best = max(best, q)
        U_new[s] = best
        delta = max(delta, abs(best - U[s]))
    U = U_new
    if it <= 5 or it % 10 == 0 or delta < 1e-4:
        print(f"{it:>5}{U[(1,1)]:>10.4f}{U[(1,3)]:>10.4f}{U[(3,3)]:>10.4f}{U[(4,1)]:>10.4f}{delta:>13.6f}")
    if delta < 1e-4:
        print(f"\nConverged after {it} iterations (max change < 1e-4)")
        break

# ---- Final utilities ----
print("\n=== Converged utility values U(s) ===\n")
for r in range(rows, 0, -1):
    line = f"  row {r} |"
    for c in range(1, cols+1):
        line += "   WALL " if (c, r) == WALL else f"{U[(c,r)]:>8.4f}"
    print(line)
print("          " + "".join(f"{'col '+str(c):>8}" for c in range(1, cols+1)))

# ---- Extract the optimal policy ----
print("\n=== Optimal policy (best action in every state) ===\n")
ARROW = {"Up": "^", "Down": "v", "Left": "<", "Right": ">"}
policy = {}
for s in states:
    if s in TERMINALS: continue
    qs = {a: sum(p * (R_STEP + GAMMA * U[ns]) for p, ns in transitions(s, a))
          for a in ACTIONS}
    policy[s] = max(qs, key=qs.get)

for r in range(rows, 0, -1):
    line = f"  row {r} |"
    for c in range(1, cols+1):
        if (c, r) == WALL:              line += "   #  "
        elif (c, r) in TERMINALS:       line += f"  {'+1' if TERMINALS[(c,r)]>0 else '-1':>2}  "
        else:                           line += f"   {ARROW[policy[(c,r)]]}  "
    print(line)
print("          " + "".join(f"{'col '+str(c):>6}" for c in range(1, cols+1)))

print("\nThe policy routes the agent up and around the wall to reach +1 while")
print("avoiding the -1 square, just as expected.")
print(f"\nUtility of the START state (1,1): {U[(1,1)]:.4f}")
print("Sanity check: reaching +1 in 5 steps gives 4*(-0.04) + 1 = 0.84,")
print("and the converged utility is lower because of the stochastic slips.")

=== Value Iteration on the 4x3 Grid World ===

Grid (column, row), (1,1) is bottom-left:

     3 | . . . +1
     2 | . # . -1        # = wall/forbidden square (2,2)
     1 | S . . .         S = START
       +---------
       1 2 3 4

Stochastic environment: the intended action succeeds with p=0.8,
and moves at right angles with p=0.1 each. Bumping into a wall
leaves the agent in the same square.
Reward: -0.04 for every non-terminal transition. Discount gamma=1.0

Bellman update:  U(s) <- max_a  sum_s' P(s'|s,a) [ R(s,a,s') + gamma*U(s') ]

 iter    U(1,1)    U(1,3)    U(3,3)    U(4,1)   max change
----------------------------------------------------------
    1   -0.0400   -0.0400    0.7600   -0.0400     0.760000
    2   -0.0800   -0.0800    0.8320   -0.0800     0.600000
    3   -0.1200    0.3920    0.8896   -0.1200     0.472000
    4   -0.1600    0.5773    0.9062    0.1002     0.369600
    5    0.1625    0.6980    0.9135    0.1849     0.322496
   10    0.6754    0.8090    0.9178    0.

## Example 2: Q-Learning on the Same Grid World

Value iteration needed the transition model; Q-learning does not. The agent learns purely from experience using the temporal-difference update and an epsilon-greedy policy that explores early and exploits later. After 8000 episodes it recovers the SAME optimal policy and approaches the same state values, without ever being told the transition probabilities.

In [2]:
# Chapter 33 - Example 2: Q-Learning on the same 4x3 Grid World
import numpy as np
np.random.seed(0)

print("=== Q-Learning: Model-Free Reinforcement Learning ===\n")
print("Value iteration NEEDS the transition model P(s'|s,a) and reward function.")
print("Q-Learning does NOT: the agent learns purely from experience by")
print("interacting with the environment.\n")
print("Update rule (temporal-difference):")
print("  Q(s,a) <- Q(s,a) + alpha [ r + gamma*max_a' Q(s',a') - Q(s,a) ]")
print("            \\____________________ TD error ____________________/\n")

rows, cols = 3, 4
WALL      = (2, 2)
TERMINALS = {(4, 3): +1.0, (4, 2): -1.0}
R_STEP, GAMMA, ALPHA = -0.04, 1.0, 0.1
ACTIONS = ["Up", "Down", "Left", "Right"]
DELTA   = {"Up": (0, 1), "Down": (0, -1), "Left": (-1, 0), "Right": (1, 0)}
PERP    = {"Up": ["Left","Right"], "Down": ["Left","Right"],
           "Left": ["Up","Down"],  "Right": ["Up","Down"]}
states  = [(c, r) for c in range(1, cols+1) for r in range(1, rows+1) if (c, r) != WALL]

def move(s, a):
    dc, dr = DELTA[a]; ns = (s[0]+dc, s[1]+dr)
    return s if (ns == WALL or not (1 <= ns[0] <= cols and 1 <= ns[1] <= rows)) else ns

def step(s, a):
    """The ENVIRONMENT: stochastic outcome, agent does not know these probabilities."""
    u = np.random.rand()
    actual = a if u < 0.8 else (PERP[a][0] if u < 0.9 else PERP[a][1])
    ns = move(s, actual)
    r  = TERMINALS[ns] if ns in TERMINALS else R_STEP
    return ns, r, ns in TERMINALS

Q = {(s, a): 0.0 for s in states for a in ACTIONS}

def eps_greedy(s, eps):
    if np.random.rand() < eps:
        return ACTIONS[np.random.randint(4)]          # EXPLORE
    qs = [Q[(s, a)] for a in ACTIONS]
    return ACTIONS[int(np.argmax(qs))]                 # EXPLOIT

EPISODES = 8000
print(f"Training for {EPISODES} episodes with epsilon-greedy exploration")
print("(epsilon decays from 1.0 to 0.05: explore early, exploit later)\n")
print(f"{'episode':>9}{'epsilon':>10}{'Q(1,1) best':>14}{'avg return':>13}")
print("-" * 48)

returns = []
for ep in range(1, EPISODES + 1):
    eps = max(0.05, 1.0 - ep / (EPISODES * 0.6))
    s, total, done, steps = (1, 1), 0.0, False, 0
    while not done and steps < 100:
        a = eps_greedy(s, eps)
        ns, r, done = step(s, a)
        best_next = 0.0 if done else max(Q[(ns, a2)] for a2 in ACTIONS)
        Q[(s, a)] += ALPHA * (r + GAMMA * best_next - Q[(s, a)])
        s = ns; total += r; steps += 1
    returns.append(total)
    if ep % 1600 == 0:
        print(f"{ep:>9}{eps:>10.3f}{max(Q[((1,1),a)] for a in ACTIONS):>14.4f}"
              f"{np.mean(returns[-500:]):>13.4f}")

print("\n=== Learned Q-values for the START state (1,1) ===\n")
for a in ACTIONS:
    bar = "#" * int(max(0, Q[((1,1), a)]) * 40)
    print(f"  Q((1,1), {a:<5}) = {Q[((1,1),a)]:+.4f}  {bar}")
best_a = max(ACTIONS, key=lambda a: Q[((1,1), a)])
print(f"  -> best action from START: {best_a}")

print("\n=== Learned policy (from Q-values only, no model used) ===\n")
ARROW = {"Up":"^","Down":"v","Left":"<","Right":">"}
for r in range(rows, 0, -1):
    line = f"  row {r} |"
    for c in range(1, cols+1):
        s = (c, r)
        if s == WALL:            line += "   #  "
        elif s in TERMINALS:     line += f"  {'+1' if TERMINALS[s]>0 else '-1':>2}  "
        else:
            a = max(ACTIONS, key=lambda a: Q[(s, a)])
            line += f"   {ARROW[a]}  "
    print(line)
print("          " + "".join(f"{'col '+str(c):>6}" for c in range(1, cols+1)))

print("\n=== V(s) = max_a Q(s,a)  compared with value iteration ===\n")
vi = {(1,1):0.7053, (1,3):0.8116, (3,3):0.9178, (4,1):0.3878}
print(f"  {'state':<8}{'Q-learning':>12}{'value iteration':>18}{'difference':>13}")
print("  " + "-" * 52)
for s, u in vi.items():
    v = max(Q[(s, a)] for a in ACTIONS)
    print(f"  {str(s):<8}{v:>12.4f}{u:>18.4f}{abs(v-u):>13.4f}")

print("\nQ-learning approaches the same values WITHOUT ever being told the")
print("transition probabilities - it learned them implicitly from experience.")

=== Q-Learning: Model-Free Reinforcement Learning ===

Value iteration NEEDS the transition model P(s'|s,a) and reward function.
Q-Learning does NOT: the agent learns purely from experience by
interacting with the environment.

Update rule (temporal-difference):
  Q(s,a) <- Q(s,a) + alpha [ r + gamma*max_a' Q(s',a') - Q(s,a) ]
            \____________________ TD error ____________________/

Training for 8000 episodes with epsilon-greedy exploration
(epsilon decays from 1.0 to 0.05: explore early, exploit later)

  episode   epsilon   Q(1,1) best   avg return
------------------------------------------------
     1600     0.667        0.7664      -0.3023
     3200     0.333        0.7082       0.4222
     4800     0.050        0.7732       0.6987
     6400     0.050        0.7679       0.7361
     8000     0.050        0.7681       0.7194

=== Learned Q-values for the START state (1,1) ===

  Q((1,1), Up   ) = +0.7681  ##############################
  Q((1,1), Down ) = +0.6831  ########

## Example 3: Exploration vs Exploitation, and TD(0)

The bandit experiment quantifies the central RL dilemma: pure exploitation never finds the best arm, pure exploration never capitalizes on what it learns, and a moderate epsilon does best. A decaying epsilon then identifies the optimal arm reliably. The final section runs TD(0) on a chain, showing values propagating backwards from the goal - the bootstrapping idea behind Q-learning and deep RL.

In [3]:
# Chapter 33 - Example 3: Exploration vs Exploitation + TD(0) learning
import numpy as np
np.random.seed(42)

print("=== Exploration vs Exploitation ===\n")
print("The core RL dilemma:")
print("  EXPLOIT : take the action currently believed to be best")
print("  EXPLORE : try other actions to discover something better")
print("Too much exploitation -> stuck on a mediocre action.")
print("Too much exploration  -> wastes reward on known-bad actions.\n")

# ---- Multi-armed bandit ----
true_means = np.array([0.20, 0.55, 0.35, 0.80, 0.45])   # arm 4 is best
K, STEPS, RUNS = len(true_means), 1500, 200
print(f"Bandit with {K} actions. True mean rewards: {true_means}")
print(f"Optimal action = arm {int(np.argmax(true_means))+1} (mean {true_means.max():.2f})\n")

def run_bandit(eps, steps=STEPS):
    Q, N = np.zeros(K), np.zeros(K)
    total, opt_count = 0.0, 0
    for t in range(steps):
        a = np.random.randint(K) if np.random.rand() < eps else int(np.argmax(Q))
        r = np.random.randn() * 0.3 + true_means[a]
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]                # incremental sample average
        total += r
        opt_count += (a == np.argmax(true_means))
    return total, opt_count / steps, Q

print(f"{'strategy':<28}{'avg reward':>12}{'% optimal action':>19}")
print("-" * 60)
results = {}
for eps, label in [(0.00, "Pure exploitation (eps=0)"),
                   (0.01, "eps = 0.01"),
                   (0.10, "eps = 0.10"),
                   (0.30, "eps = 0.30"),
                   (1.00, "Pure exploration (eps=1)")]:
    tots, opts = [], []
    for _ in range(RUNS):
        t, o, _ = run_bandit(eps)
        tots.append(t / STEPS); opts.append(o)
    results[label] = (np.mean(tots), np.mean(opts))
    print(f"  {label:<26}{np.mean(tots):>12.4f}{np.mean(opts)*100:>18.1f}%")

best = max(results, key=lambda k: results[k][0])
print(f"\n  Best strategy here: {best}")
print("  Pure exploitation can lock onto a suboptimal arm found early;")
print("  pure exploration never capitalizes on what it has learned.")
print("  A small epsilon (or a decaying one) balances the two.\n")

# ---- Epsilon decay ----
print("--- Decaying epsilon: explore early, exploit later ---\n")
Q, N = np.zeros(K), np.zeros(K)
total = 0.0
print(f"{'step':>7}{'epsilon':>10}{'best arm est.':>15}{'running avg':>13}")
print("-" * 48)
for t in range(1, STEPS + 1):
    eps = max(0.02, 1.0 / np.sqrt(t))
    a = np.random.randint(K) if np.random.rand() < eps else int(np.argmax(Q))
    r = np.random.randn() * 0.3 + true_means[a]
    N[a] += 1; Q[a] += (r - Q[a]) / N[a]; total += r
    if t in (10, 100, 500, 1000, 1500):
        print(f"{t:>7}{eps:>10.3f}{int(np.argmax(Q))+1:>15}{total/t:>13.4f}")
print(f"\n  Final estimates Q: {np.round(Q, 3)}")
print(f"  True means       : {true_means}")
print(f"  Identified best arm: {int(np.argmax(Q))+1} (true best: {int(np.argmax(true_means))+1})")

# ---- TD(0) ----
print("\n=== Temporal-Difference Learning TD(0) ===\n")
print("TD(0) updates the value of a state from the very next step, without")
print("waiting for the episode to end:")
print("  V(s) <- V(s) + alpha [ r + gamma*V(s') - V(s) ]\n")

chain = ["A", "B", "C", "D", "GOAL"]
V = {s: 0.0 for s in chain}
V["GOAL"] = 0.0
alpha, gamma = 0.1, 0.9
rewards = {"A":-0.1, "B":-0.1, "C":-0.1, "D":+1.0}

print("Simple chain A -> B -> C -> D -> GOAL (reward +1 at the end)\n")
print(f"{'episode':>9}" + "".join(f"{s:>9}" for s in chain[:-1]))
print("-" * 46)
for ep in range(1, 51):
    for i, s in enumerate(chain[:-1]):
        s_next = chain[i+1]
        r = rewards[s]
        V[s] += alpha * (r + gamma * V[s_next] - V[s])
    if ep in (1, 5, 10, 25, 50):
        print(f"{ep:>9}" + "".join(f"{V[s]:>9.4f}" for s in chain[:-1]))

print("\n  Values propagate BACKWARD from the goal: D learns first, then C, B, A.")
print("  This bootstrapping (learning a guess from a guess) is what makes TD")
print("  methods work online, without waiting for the final outcome.")
print("\nDeep RL replaces the Q-table with a neural network (Deep Q-Network),")
print("which is what allows RL to scale to games, robots and self-driving cars.")

=== Exploration vs Exploitation ===

The core RL dilemma:
  EXPLOIT : take the action currently believed to be best
  EXPLORE : try other actions to discover something better
Too much exploitation -> stuck on a mediocre action.
Too much exploration  -> wastes reward on known-bad actions.

Bandit with 5 actions. True mean rewards: [0.2  0.55 0.35 0.8  0.45]
Optimal action = arm 4 (mean 0.80)

strategy                      avg reward   % optimal action
------------------------------------------------------------
  Pure exploitation (eps=0)       0.3201               0.0%
  eps = 0.01                      0.6665              60.3%
  eps = 0.10                      0.7558              88.8%
  eps = 0.30                      0.6967              74.8%
  Pure exploration (eps=1)        0.4706              20.0%

  Best strategy here: eps = 0.10
  Pure exploitation can lock onto a suboptimal arm found early;
  pure exploration never capitalizes on what it has learned.
  A small epsilon (or a d